# 01 · The pre-training objective: next-token prediction

The objective every LLM starts with: given tokens so far, predict the next one.
**Every token counts equally (weight = 1).**

Note: Tinker trains LoRA adapters on an existing model, so this is technically
*continued* pre-training. The objective is identical to pre-training from scratch.

In [1]:
import urllib.request

from common import *
from tinker_cookbook.hyperparam_utils import get_lr

text = urllib.request.urlopen(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
).read().decode()
all_tokens = tokenizer.encode(text, add_special_tokens=False)
print(f"{len(text):,} chars -> {len(all_tokens):,} tokens")

/home/ray/default/post-training-webinar/.venv/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (326355 > 262144). Running this sequence through the model will result in indexing errors


1,115,394 chars -> 326,355 tokens


In [2]:
SEQ_LEN, BATCH_SIZE, STEPS = 256, 16, 30
LR = get_lr(MODEL)
chunks = [all_tokens[i : i + SEQ_LEN] for i in range(0, len(all_tokens) - SEQ_LEN, SEQ_LEN)]

In [3]:
# ── LIVE ──────────────────────────────────────────────────────────────────
def pretrain_datum(tokens):
    weights = [1.0] * len(tokens)  # <- the whole idea: every token is a training signal
    return make_datum(tokens, weights)

In [ ]:
# ── LIVE: the training loop (we'll reuse this shape all day) ───────────────
training_client = new_training_client()

for step in range(STEPS):
    batch = [pretrain_datum(c) for c in chunks[step * BATCH_SIZE : (step + 1) * BATCH_SIZE]]
    fwd_bwd = training_client.forward_backward(batch, loss_fn="cross_entropy")
    optim = training_client.optim_step(adam(LR))
    # Both calls return futures: they're queued back-to-back, then we wait.
    print(f"step {step:2d}  nll/token = {mean_nll(fwd_bwd.result(), batch):.3f}")
    optim.result()

In [ ]:
sampling_client = training_client.save_weights_and_get_sampling_client().result()
show("after pre-training on Shakespeare", generate(sampling_client, "ROMEO:", chat=False, temperature=0.7, max_tokens=120))

**Takeaway:** pre-training teaches the model *the distribution of text*. Nothing
about following instructions or getting the right answer. That's what post-training is for.